# Ele — Phase 1 on Google Colab

Frozen SigLIP + linear head on CIFAKE (no augmentation).

## How to run

1. Open [Google Colab](https://colab.research.google.com/).
2. **File → Upload notebook** and choose this file (`notebooks/colab_phase1.ipynb`), or **File → Open notebook → GitHub** after you push it.
3. **Runtime → Change runtime type → T4 GPU → Save**.
4. **Runtime → Run all** (or click the play button on each cell top to bottom).

Training is ~2–4 hours on a free T4 for 3 epochs on full CIFAKE. If you hit CUDA out of memory, rerun the train cell with `--batch-size 8`.

Colab will delete local files when the runtime dies. Run the last cell to copy `models/phase1.pt` to Drive.

## 1. Confirm GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Restart session."
)
print(torch.cuda.get_device_name(0))
print("CUDA", torch.version.cuda)

## 2. Clone the repo

Uses GitHub. If you have unpushed local fixes, push `main` first, or skip this cell and upload the `Ele` folder to Drive and `%cd` into it.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/harikrishn4a/ele.git"
ROOT = Path("/content/ele")

if not ROOT.exists():
    !git clone --depth 1 {REPO} {ROOT}
else:
    print("Already cloned:", ROOT)

%cd /content/ele
!git log -1 --oneline

## 3. Install packages

Colab already has PyTorch with CUDA. This adds OpenCLIP (SigLIP) and the rest of Ele.

In [ ]:
!pip install -q open-clip-torch timm transformers scikit-learn opencv-python einops peft kagglehub tensorboard tqdm pandas Pillow

## 4. Patch GitHub `main` if needed

Older commits import a missing `evaluate_model` and look for CIFAKE `synthetic/` instead of `FAKE/`.

In [ ]:
from pathlib import Path

train_py = Path("src/train.py")
text = train_py.read_text()
old = "from src.evaluate import evaluate_model"
if old in text:
    train_py.write_text(text.replace(old, "# " + old, 1))
    print("Patched src/train.py import")
else:
    print("train.py already ok")

## 5. Download CIFAKE and fill `data/`

Copies official CIFAKE `train`/`test` + `REAL`/`FAKE` (not a 70/30 split).

In [ ]:
import shutil
from pathlib import Path

import kagglehub

cifake = Path(
    kagglehub.dataset_download(
        "birdy654/cifake-real-and-ai-generated-synthetic-images"
    )
)
print("CIFAKE at", cifake)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def first_dir(parent, names):
    for name in names:
        p = parent / name
        if p.is_dir():
            return p
    return None


def copy_images(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    files = [p for p in src.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    for src_file in files:
        out = dst / src_file.name
        if not out.exists():
            shutil.copy2(src_file, out)
    print(f"{len(files):>6}  {src} -> {dst}")


root = Path("data")
for split in ("train", "test"):
    split_dir = cifake / split
    copy_images(first_dir(split_dir, ["REAL", "real"]), root / split / "real")
    copy_images(first_dir(split_dir, ["FAKE", "fake", "synthetic"]), root / split / "fake")

for split in ("train", "test"):
    n_real = len(list((root / split / "real").glob("*")))
    n_fake = len(list((root / split / "fake").glob("*")))
    print(f"{split}: {n_real} real, {n_fake} fake")
    assert n_real > 0 and n_fake > 0, f"Empty {split} split"

## 6. Hugging Face token (optional)

Only if the SigLIP download is rate-limited. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os

# os.environ["HF_TOKEN"] = "hf_..."
print("HF_TOKEN set:" , bool(os.environ.get("HF_TOKEN")))

## 7. Train Phase 1

This is the command. Colab uses CUDA automatically. First run downloads ~3.5GB of SigLIP weights, then trains 3 epochs.

Keep this tab open. Colab disconnects if the browser sleeps for a long time.

In [ ]:
!python3 -m src.train \
  --backbone sigclip \
  --epochs 3 \
  --batch-size 16 \
  --output models/phase1.pt

## 8. Evaluate transform grid

In [ ]:
!python3 -m src.evaluate \
  --model models/phase1.pt \
  --backbone sigclip \
  --dataset data \
  --output results/phase1.csv

## 9. Save checkpoint to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
out = Path("/content/drive/MyDrive/Ele")
out.mkdir(parents=True, exist_ok=True)

for src in (Path("models/phase1.pt"), Path("results/phase1.csv")):
    if src.exists():
        shutil.copy2(src, out / src.name)
        print("Copied", src, "->", out / src.name)
    else:
        print("Missing", src)